# Онтология Дара — LoRA Fine-tuning

**Модель**: `Qwen/Qwen2.5-3B-Instruct` (adam, eva, bezalel) + `Qwen/Qwen2.5-0.5B-Instruct` (serafim)

**Метод**: LoRA (Unsloth) — только дообучаем адаптеры, веса не трогаем

**Датасет**: 93+ пары `{instruction, input, output, system}` из онтологии

**Результат**: GGUF Q4_K_M → загрузить в Ollama

---
**Запуск**: Runtime → Change runtime type → T4 GPU → Run all


In [ ]:
# Ячейка 1: Установка зависимостей
# Unsloth — fastest LoRA for LLMs (2x speed, 70% less VRAM)
!pip install unsloth==2024.12.4 -q
!pip install trl datasets transformers accelerate bitsandbytes -q
!pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' -q
print('✓ Установлено')

In [ ]:
# Ячейка 2: Конфигурация
import os

# ── Выбор агента для обучения ──────────────────────────────────────────────
AGENT = 'adam'         # 'adam' | 'eva' | 'serafim' | 'bezalel' | 'all'

# ── Параметры моделей ──────────────────────────────────────────────────────
MODEL_MAP = {
    'adam':    'unsloth/Qwen2.5-3B-Instruct-bnb-4bit',
    'eva':     'unsloth/Qwen2.5-3B-Instruct-bnb-4bit',
    'bezalel': 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit',
    'serafim': 'unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit',
    'all':     'unsloth/Qwen2.5-3B-Instruct-bnb-4bit',  # общий
}
MODEL_NAME = MODEL_MAP[AGENT]

# ── LoRA параметры ─────────────────────────────────────────────────────────
LORA_CONFIG = {
    'r': 16,                        # rank адаптера (16 = хороший баланс)
    'lora_alpha': 32,               # обычно 2*r
    'lora_dropout': 0.0,            # Unsloth рекомендует 0
    'target_modules': [             # какие слои адаптируем
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    'bias': 'none',
    'use_gradient_checkpointing': 'unsloth',
    'random_state': 42,
}

# ── Обучение ───────────────────────────────────────────────────────────────
TRAIN_CONFIG = {
    'num_train_epochs': 3,
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 4,
    'learning_rate': 2e-4,
    'fp16': True,
    'bf16': False,
    'max_seq_length': 2048,
    'warmup_ratio': 0.1,
    'lr_scheduler_type': 'cosine',
    'output_dir': f'./outputs/{AGENT}',
    'save_strategy': 'epoch',
    'logging_steps': 10,
    'seed': 42,
}

print(f'✓ Конфигурация: агент={AGENT}, модель={MODEL_NAME}')
print(f'  LoRA rank={LORA_CONFIG["r"]}, epochs={TRAIN_CONFIG["num_train_epochs"]}')

In [ ]:
# Ячейка 3: Загрузка датасета
# Вариант A: загрузить файл вручную
from google.colab import files
import json

print('Загрузи файл датасета:')
print(f'  Файл: data/finetune/{AGENT}.jsonl  (или all_agents.jsonl)')
print('  Нажми кнопку «Выбрать файлы» ниже')
uploaded = files.upload()

dataset_file = list(uploaded.keys())[0]
with open(dataset_file) as f:
    raw_data = [json.loads(line) for line in f if line.strip()]

print(f'\n✓ Загружено {len(raw_data)} примеров из {dataset_file}')
print('Пример:')
print(json.dumps(raw_data[0], ensure_ascii=False, indent=2)[:400])

In [ ]:
# Ячейка 3b: Альтернатива — загрузка из Google Drive
# Раскомментируй если датасет уже на Drive

# from google.colab import drive
# drive.mount('/content/drive')
# import json
# DRIVE_PATH = '/content/drive/MyDrive/gift-ontology/data/finetune/'
# dataset_file = f'{DRIVE_PATH}{AGENT}.jsonl'
# with open(dataset_file) as f:
#     raw_data = [json.loads(line) for line in f if line.strip()]
# print(f'✓ Загружено {len(raw_data)} примеров')

In [ ]:
# Ячейка 4: Загрузка модели + LoRA
from unsloth import FastLanguageModel
import torch

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=TRAIN_CONFIG['max_seq_length'],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_CONFIG['r'],
    target_modules=LORA_CONFIG['target_modules'],
    lora_alpha=LORA_CONFIG['lora_alpha'],
    lora_dropout=LORA_CONFIG['lora_dropout'],
    bias=LORA_CONFIG['bias'],
    use_gradient_checkpointing=LORA_CONFIG['use_gradient_checkpointing'],
    random_state=LORA_CONFIG['random_state'],
)

model.print_trainable_parameters()
print('✓ Модель загружена с LoRA адаптерами')

In [ ]:
# Ячейка 5: Подготовка датасета
from datasets import Dataset

# Alpaca шаблон с системным промптом
ALPACA_TEMPLATE = """Ниже — инструкция, описывающая задачу. Напиши ответ.

### Система:
{system}

### Инструкция:
{instruction}

### Ввод:
{input}

### Ответ:
{output}"""

EOS_TOKEN = tokenizer.eos_token

def format_sample(sample):
    text = ALPACA_TEMPLATE.format(
        system=sample.get('system', ''),
        instruction=sample.get('instruction', ''),
        input=sample.get('input', ''),
        output=sample.get('output', ''),
    ) + EOS_TOKEN
    return {'text': text}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_sample)

# 90/10 split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f'✓ Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')
print('\nПример форматированного текста:')
print(train_dataset[0]['text'][:600])

In [ ]:
# Ячейка 6: Обучение
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field='text',
    max_seq_length=TRAIN_CONFIG['max_seq_length'],
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=TRAIN_CONFIG['per_device_train_batch_size'],
        gradient_accumulation_steps=TRAIN_CONFIG['gradient_accumulation_steps'],
        warmup_ratio=TRAIN_CONFIG['warmup_ratio'],
        num_train_epochs=TRAIN_CONFIG['num_train_epochs'],
        learning_rate=TRAIN_CONFIG['learning_rate'],
        fp16=TRAIN_CONFIG['fp16'],
        bf16=TRAIN_CONFIG['bf16'],
        logging_steps=TRAIN_CONFIG['logging_steps'],
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type=TRAIN_CONFIG['lr_scheduler_type'],
        seed=TRAIN_CONFIG['seed'],
        output_dir=TRAIN_CONFIG['output_dir'],
        save_strategy=TRAIN_CONFIG['save_strategy'],
        eval_strategy='epoch',
        load_best_model_at_end=True,
        report_to='none',
    ),
)

print('Начинаем обучение...')
trainer_stats = trainer.train()
print(f'\n✓ Обучение завершено!')
print(f'  Потеря: {trainer_stats.training_loss:.4f}')
print(f'  Время: {trainer_stats.metrics["train_runtime"]:.0f} сек')

In [ ]:
# Ячейка 7: Тест обученной модели
FastLanguageModel.for_inference(model)

TEST_PROMPTS = {
    'adam':    'В матрице W лицо «bezalel» не имеет входящих нитей.',
    'eva':     'Предложение: добавить метрику лайков в боте.',
    'serafim': 'батарея 8% до базы 6 минут',
    'bezalel': 'Напиши функцию kenosis(agent) — кенозис агента в Node.js',
    'all':     'В матрице W лицо «bezalel» не имеет входящих нитей.',
}

test_prompt = TEST_PROMPTS.get(AGENT, TEST_PROMPTS['adam'])
formatted = ALPACA_TEMPLATE.format(
    system=f'Ты агент онтологии дара ({AGENT})',
    instruction=test_prompt,
    input='',
    output='',
).replace('\n### Ответ:\n', '\n### Ответ:\n')  # убрать пустой output

inputs = tokenizer(formatted, return_tensors='pt').to('cuda')
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.6,
        top_p=0.9,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'Промпт: {test_prompt}')
print(f'\nОтвет {AGENT}:')
print(response)

In [ ]:
# Ячейка 8: Экспорт в GGUF (для Ollama)
GGUF_DIR = f'./gguf/{AGENT}'

print('Экспортируем в GGUF Q4_K_M...')
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method='q4_k_m',  # лучший баланс качество/размер
)

import os
gguf_files = [f for f in os.listdir(GGUF_DIR) if f.endswith('.gguf')]
print(f'✓ GGUF файлы: {gguf_files}')
for f in gguf_files:
    size = os.path.getsize(os.path.join(GGUF_DIR, f)) / 1e9
    print(f'  {f}: {size:.2f} GB')

In [ ]:
# Ячейка 9: Создать Modelfile для Ollama
SYSTEM_PROMPTS = {
    'adam': open('/content/Modelfile.adam').read().split('SYSTEM """')[1].split('"""')[0] if os.path.exists('/content/Modelfile.adam') else 'Ты Адам — агент онтологии дара.',
    'eva':  'Ты Ева — точильный камень Адама. Проверяешь и усиливаешь каждый дар.',
    'serafim': 'Ты Серафим — ИИ на борту дрона. Говоришь кратко. L1: одно слово. L2: 1-2 предложения.',
    'bezalel': 'Ты Веселеил — строитель кода. Только рабочий Node.js/ESM код. Никакой философии.',
    'all': 'Ты агент онтологии дара. Дар необратим. Кенозис реален. Анамнезис — не архив.',
}

gguf_file = gguf_files[0] if gguf_files else 'model-q4_k_m.gguf'
modelfile_content = f"""FROM ./{gguf_file}

PARAMETER temperature {'0.2' if AGENT in ['serafim', 'bezalel'] else '0.6'}
PARAMETER top_p {'0.9' if AGENT in ['serafim', 'bezalel'] else '0.88'}
PARAMETER repeat_penalty 1.1
PARAMETER num_predict {'60' if AGENT == 'serafim' else '256' if AGENT == 'adam' else '512'}

SYSTEM \"\"\"{SYSTEM_PROMPTS.get(AGENT, '')}\"\"\"
"""

modelfile_path = f'/content/Modelfile.{AGENT}-lora'
with open(modelfile_path, 'w') as f:
    f.write(modelfile_content)

print(f'✓ Modelfile создан: {modelfile_path}')
print('\nПервые 200 символов:')
print(modelfile_content[:200])

In [ ]:
# Ячейка 10: Скачать результаты
import zipfile, os
from google.colab import files

ZIP_NAME = f'gift-{AGENT}-lora.zip'

with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zf:
    # GGUF файлы
    for fname in os.listdir(GGUF_DIR):
        if fname.endswith('.gguf'):
            zf.write(os.path.join(GGUF_DIR, fname), fname)
    # Modelfile
    zf.write(modelfile_path, f'Modelfile.{AGENT}-lora')

zip_size = os.path.getsize(ZIP_NAME) / 1e9
print(f'✓ Архив: {ZIP_NAME} ({zip_size:.2f} GB)')
print('\nСкачиваю...')
files.download(ZIP_NAME)

print('\n=== После скачивания ===')
print(f'На локальной машине (WSL2):')
print(f'  # Распаковать в data/lora/{AGENT}/')
print(f'  # Зарегистрировать в Ollama:')
print(f'  ollama create {AGENT}-lora -f data/lora/{AGENT}/Modelfile.{AGENT}-lora')
print(f'  ollama run {AGENT}-lora "тест"')

In [ ]:
# Ячейка 11: Сохранить в Google Drive (опционально, для постоянного хранения)
# Раскомментируй если хочешь хранить на Drive

# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# DRIVE_OUT = f'/content/drive/MyDrive/gift-ontology/lora/{AGENT}/'
# os.makedirs(DRIVE_OUT, exist_ok=True)
# for fname in os.listdir(GGUF_DIR):
#     if fname.endswith('.gguf'):
#         shutil.copy(os.path.join(GGUF_DIR, fname), DRIVE_OUT)
# shutil.copy(modelfile_path, DRIVE_OUT)
# print(f'✓ Сохранено в Drive: {DRIVE_OUT}')

---
## Альтернатива: Kaggle Notebooks (30 ч/неделю T4 бесплатно)

Kaggle даёт больше GPU-часов в неделю, чем Colab Free.

1. Перейди на [kaggle.com/code](https://www.kaggle.com/code) → New Notebook
2. Settings → Accelerator → GPU T4 x2
3. Загрузи этот ноутбук или скопируй ячейки
4. Dataset: Add data → Upload → загрузи `data/finetune/*.jsonl`

Kaggle API для автоматического запуска:
```bash
pip install kaggle
kaggle kernels push -p ./colab/  # запустить обучение через API
kaggle kernels output username/gift-finetune  # скачать результаты
```

## Modal.com (платный, но $30 кредиты бесплатно)

```python
# modal_finetune.py — запустить одной командой: modal run modal_finetune.py
import modal
app = modal.App('gift-finetune')

@app.function(gpu='T4', timeout=3600)
def finetune():
    # ... код обучения здесь ...
    pass
```
